In [8]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124
Note: you may need to restart the kernel to use updated packages.


# BiLSTM 1

In [ ]:
from pathlib import Path
import json
import gc
import random
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import f1_score, confusion_matrix, classification_report

# =========================================================
# 0) Reproducibility
# =========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# =========================================================
# 1) Paths
# =========================================================
FEATURE_ROOT = Path(r"C:\SeniorProject\Good_Data\NPZ_SPLITS_FILTERED_CAP1H_IMPD\CNN_FEATURES_FOR_BILSTM")

TRAIN_ROOT = FEATURE_ROOT / "TRAIN"
VAL_ROOT   = FEATURE_ROOT / "VAL"
TEST_ROOT  = FEATURE_ROOT / "TEST"

OUT_ROOT = FEATURE_ROOT.parent / "BILSTM_RESULTS_WEIGHTED_CE"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

BEST_CKPT_PATH = OUT_ROOT / "best_model.pt"
LAST_CKPT_PATH = OUT_ROOT / "last_model.pt"
METRICS_JSON   = OUT_ROOT / "final_metrics.json"
CLS_REPORT_TXT = OUT_ROOT / "test_classification_report.txt"
CM_NPY         = OUT_ROOT / "test_confusion_matrix.npy"
PRED_NPZ       = OUT_ROOT / "test_predictions.npz"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================================================
# 2) Config
# =========================================================
NUM_CLASSES = 3

SEQ_LEN = 10
SEQ_STRIDE = 1

BATCH_SIZE = 256
MAX_EPOCHS = 60
MIN_EPOCHS = 12
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-4
CLIP_NORM = 1.0

HIDDEN_SIZE = 128
NUM_LAYERS = 2
DROPOUT = 0.3

NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

# =========================================================
# 3) Helpers
# =========================================================
def load_feature_run(path: Path):
    npz = np.load(path, allow_pickle=True)
    return np.asarray(npz["F"], dtype=np.float32), \
           np.asarray(npz["y"], dtype=np.int64), \
           np.asarray(npz["win_idx"], dtype=np.int32), \
           str(npz["base"][0]) if "base" in npz else path.stem


def compute_train_feature_stats(train_root: Path):
    sum_vec, sumsq_vec, total_n = None, None, 0

    for p in sorted(train_root.glob("*.npz")):
        F, _, _, _ = load_feature_run(p)
        if F.ndim != 2 or F.shape[0] == 0:
            continue

        if sum_vec is None:
            sum_vec = F.sum(axis=0, dtype=np.float64)
            sumsq_vec = (F**2).sum(axis=0, dtype=np.float64)
        else:
            sum_vec += F.sum(axis=0)
            sumsq_vec += (F**2).sum(axis=0)

        total_n += F.shape[0]

    mean = sum_vec / total_n
    var = (sumsq_vec / total_n) - (mean**2)
    std = np.sqrt(np.maximum(var, 1e-8))

    return mean.astype(np.float32), std.astype(np.float32)


def compute_sequence_class_weights(root, seq_len, stride):
    counts = np.zeros(NUM_CLASSES)

    for p in root.glob("*.npz"):
        _, y, _, _ = load_feature_run(p)
        for i in range(len(y) - seq_len + 1):
            counts[y[i + seq_len - 1]] += 1

    counts = np.maximum(counts, 1)
    weights = counts.sum() / (NUM_CLASSES * counts)
    weights /= weights.mean()

    return counts, weights.astype(np.float32)

# =========================================================
# Dataset
# =========================================================
class DatasetSeq(Dataset):
    def __init__(self, root, seq_len, stride, mean, std):
        self.samples = []
        self.seq_len = seq_len
        self.mean = mean
        self.std = std

        for p in root.glob("*.npz"):
            F, y, win_idx, base = load_feature_run(p)

            if len(F) < seq_len:
                continue

            for i in range(len(F) - seq_len + 1):
                self.samples.append((p, i))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        p, s = self.samples[idx]
        F, y, win_idx, base = load_feature_run(p)

        X = F[s:s+self.seq_len]
        X = (X - self.mean) / self.std
        X = np.nan_to_num(X)

        return torch.tensor(X), torch.tensor(y[s+self.seq_len-1])

# =========================================================
# Model
# =========================================================
class BiLSTM(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.lstm = nn.LSTM(dim, HIDDEN_SIZE, NUM_LAYERS,
                            batch_first=True, bidirectional=True)
        self.fc = nn.Linear(HIDDEN_SIZE*2, NUM_CLASSES)

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        h = torch.cat([h[-2], h[-1]], dim=1)
        return self.fc(h)

# =========================================================
# Train
# =========================================================
def evaluate(model, loader):
    model.eval()
    ys, ps = [], []

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            pred = model(X).argmax(1)
            ys.append(y.cpu().numpy())
            ps.append(pred.cpu().numpy())

    y_true = np.concatenate(ys)
    y_pred = np.concatenate(ps)

    return f1_score(y_true, y_pred, average="macro"), y_true, y_pred

# =========================================================
# MAIN
# =========================================================
def main():
    mean, std = compute_train_feature_stats(TRAIN_ROOT)

    train_ds = DatasetSeq(TRAIN_ROOT, SEQ_LEN, SEQ_STRIDE, mean, std)
    val_ds   = DatasetSeq(VAL_ROOT, SEQ_LEN, SEQ_STRIDE, mean, std)
    test_ds  = DatasetSeq(TEST_ROOT, SEQ_LEN, SEQ_STRIDE, mean, std)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE)
    test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE)

    counts, weights = compute_sequence_class_weights(TRAIN_ROOT, SEQ_LEN, SEQ_STRIDE)
    print("Weights:", weights)

    loss_fn = nn.CrossEntropyLoss(
        weight=torch.tensor(weights).to(DEVICE)
    )

    model = BiLSTM(len(mean)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)

    best = 0

    for e in range(MAX_EPOCHS):
        model.train()
        for X, y in train_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)

            opt.zero_grad()
            loss = loss_fn(model(X), y)
            loss.backward()
            opt.step()

        val_f1, _, _ = evaluate(model, val_loader)
        print(e, val_f1)

        if val_f1 > best:
            best = val_f1
            torch.save(model.state_dict(), BEST_CKPT_PATH)

    model.load_state_dict(torch.load(BEST_CKPT_PATH))

    test_f1, y_true, y_pred = evaluate(model, test_loader)

    print("Test F1:", test_f1)
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred))

if __name__ == "__main__":
    main()

CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
VRAM (GB): 7.96
Torch: 2.10.0+cu128
Feature dim: 256
Train sequences: 1838660
Val sequences  : 153833
Test sequences : 1409593
TRAIN sequence label counts: {0: 1700685, 1: 126935, 2: 11040}
Class weights: [0.10000000149011612, 0.23861843347549438, 2.7435717582702637]
Sanity logits min/max: -0.6873285174369812 0.7294876575469971
Sanity logits finite : True

[1/2] Training BiLSTM...
Epoch 001 | lr 1.00e-03 | train_loss 0.8010 | val_loss 1.0583 | val_macro_f1 0.3368
✅ Saved BEST checkpoint (val_macro_f1=0.3368, val_loss=1.0583)
Epoch 002 | lr 1.00e-03 | train_loss 0.6965 | val_loss 1.1789 | val_macro_f1 0.3565
✅ Saved BEST checkpoint (val_macro_f1=0.3565, val_loss=1.1789)
Epoch 003 | lr 1.00e-03 | train_loss 0.6456 | val_loss 1.2353 | val_macro_f1 0.3549
Epoch 004 | lr 1.00e-03 | train_loss 0.6138 | val_loss 1.2212 | val_macro_f1 0.3619
✅ Saved BEST checkpoint (val_macro_f1=0.3619, val_loss=1.2212)
Epoch 005 | lr 5.00e-04 | trai

# BiLSTM 2

In [ ]:
from pathlib import Path
import json
import gc
import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import f1_score, confusion_matrix, classification_report

# =========================================================
# 0) Reproducibility
# =========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# =========================================================
# 1) Paths
# =========================================================
FEATURE_ROOT = Path(r"C:\SeniorProject\Good_Data\NPZ_SPLITS_FILTERED_CAP1H_IMPD\CNN_FEATURES_FOR_BILSTM")

TRAIN_ROOT = FEATURE_ROOT / "TRAIN"
VAL_ROOT   = FEATURE_ROOT / "VAL"
TEST_ROOT  = FEATURE_ROOT / "TEST"

OUT_ROOT = FEATURE_ROOT.parent / "BILSTM_RESULTS_PURE_FOCAL"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

BEST_CKPT_PATH = OUT_ROOT / "best_model.pt"
LAST_CKPT_PATH = OUT_ROOT / "last_model.pt"
METRICS_JSON   = OUT_ROOT / "final_metrics.json"
CLS_REPORT_TXT = OUT_ROOT / "test_classification_report.txt"
CM_NPY         = OUT_ROOT / "test_confusion_matrix.npy"
PRED_NPZ       = OUT_ROOT / "test_predictions.npz"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================================================
# 2) Config
# =========================================================
NUM_CLASSES = 3

SEQ_LEN = 10
SEQ_STRIDE = 1

BATCH_SIZE = 256
MAX_EPOCHS = 60
MIN_EPOCHS = 12
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-4
CLIP_NORM = 1.0

HIDDEN_SIZE = 128
NUM_LAYERS = 2
DROPOUT = 0.3

FOCAL_GAMMA = 2.0

NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

# =========================================================
# 3) Helpers
# =========================================================
def load_feature_run(path: Path):
    npz = np.load(path, allow_pickle=True)
    return (
        np.asarray(npz["F"], dtype=np.float32),
        np.asarray(npz["y"], dtype=np.int64),
        np.asarray(npz["win_idx"], dtype=np.int32),
        str(npz["base"][0]) if "base" in npz else path.stem
    )

def compute_train_feature_stats(train_root: Path):
    sum_vec, sumsq_vec, total_n = None, None, 0

    for p in sorted(train_root.glob("*.npz")):
        F, _, _, _ = load_feature_run(p)
        if F.ndim != 2 or F.shape[0] == 0:
            continue

        if sum_vec is None:
            sum_vec = F.sum(axis=0)
            sumsq_vec = (F**2).sum(axis=0)
        else:
            sum_vec += F.sum(axis=0)
            sumsq_vec += (F**2).sum(axis=0)

        total_n += F.shape[0]

    mean = sum_vec / total_n
    var = (sumsq_vec / total_n) - (mean**2)
    std = np.sqrt(np.maximum(var, 1e-8))

    return mean.astype(np.float32), std.astype(np.float32)

# =========================================================
# Dataset
# =========================================================
class DatasetSeq(Dataset):
    def __init__(self, root, seq_len, stride, mean, std):
        self.samples = []
        self.seq_len = seq_len
        self.mean = mean
        self.std = std

        for p in root.glob("*.npz"):
            F, y, _, _ = load_feature_run(p)

            if len(F) < seq_len:
                continue

            for i in range(len(F) - seq_len + 1):
                self.samples.append((p, i))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        p, s = self.samples[idx]
        F, y, _, _ = load_feature_run(p)

        X = F[s:s+self.seq_len]
        X = (X - self.mean) / self.std
        X = np.nan_to_num(X)

        return torch.tensor(X), torch.tensor(y[s+self.seq_len-1])

# =========================================================
# Model
# =========================================================
class BiLSTM(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.lstm = nn.LSTM(dim, HIDDEN_SIZE, NUM_LAYERS,
                            batch_first=True, bidirectional=True)
        self.fc = nn.Linear(HIDDEN_SIZE*2, NUM_CLASSES)

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        h = torch.cat([h[-2], h[-1]], dim=1)
        return self.fc(h)

# =========================================================
# Focal Loss (NO weights)
# =========================================================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits, targets):
        log_probs = F.log_softmax(logits, dim=1)
        probs = torch.exp(log_probs)

        log_pt = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        pt = probs.gather(1, targets.unsqueeze(1)).squeeze(1)

        loss = -(1 - pt) ** self.gamma * log_pt
        return loss.mean()

# =========================================================
# Train / Eval
# =========================================================
def evaluate(model, loader):
    model.eval()
    ys, ps = [], []

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            pred = model(X).argmax(1)
            ys.append(y.cpu().numpy())
            ps.append(pred.cpu().numpy())

    y_true = np.concatenate(ys)
    y_pred = np.concatenate(ps)

    return f1_score(y_true, y_pred, average="macro"), y_true, y_pred

# =========================================================
# MAIN
# =========================================================
def main():
    mean, std = compute_train_feature_stats(TRAIN_ROOT)

    train_ds = DatasetSeq(TRAIN_ROOT, SEQ_LEN, SEQ_STRIDE, mean, std)
    val_ds   = DatasetSeq(VAL_ROOT, SEQ_LEN, SEQ_STRIDE, mean, std)
    test_ds  = DatasetSeq(TEST_ROOT, SEQ_LEN, SEQ_STRIDE, mean, std)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE)
    test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE)

    print("Train:", len(train_ds), "Val:", len(val_ds), "Test:", len(test_ds))
    print("Loss: PURE FOCAL (no weights)")

    loss_fn = FocalLoss(gamma=FOCAL_GAMMA)

    model = BiLSTM(len(mean)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)

    best = 0

    for e in range(MAX_EPOCHS):
        model.train()
        for X, y in train_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)

            opt.zero_grad()
            loss = loss_fn(model(X), y)
            loss.backward()
            opt.step()

        val_f1, _, _ = evaluate(model, val_loader)
        print(e, val_f1)

        if val_f1 > best:
            best = val_f1
            torch.save(model.state_dict(), BEST_CKPT_PATH)

    model.load_state_dict(torch.load(BEST_CKPT_PATH))

    test_f1, y_true, y_pred = evaluate(model, test_loader)

    print("Test F1:", test_f1)
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred))

if __name__ == "__main__":
    main()

CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
VRAM (GB): 7.96
Torch: 2.10.0+cu128
Feature dim: 256
Train sequences: 1838660
Val sequences  : 153833
Test sequences : 1409593
TRAIN sequence label counts: {0: 1700685, 1: 126935, 2: 11040}
Class weights: [0.10000000149011612, 0.23861843347549438, 2.7435717582702637]
Sanity logits min/max: -0.6873285174369812 0.7294876575469971
Sanity logits finite : True

[1/2] Training BiLSTM with Focal Loss...
Epoch 001 | lr 1.00e-03 | train_loss 0.0246 | val_loss 0.0551 | val_macro_f1 0.3175
✅ Saved BEST checkpoint (val_macro_f1=0.3175, val_loss=0.0551)
Epoch 002 | lr 1.00e-03 | train_loss 0.0180 | val_loss 0.0552 | val_macro_f1 0.3548
✅ Saved BEST checkpoint (val_macro_f1=0.3548, val_loss=0.0552)
Epoch 003 | lr 1.00e-03 | train_loss 0.0147 | val_loss 0.0799 | val_macro_f1 0.3558
✅ Saved BEST checkpoint (val_macro_f1=0.3558, val_loss=0.0799)
Epoch 004 | lr 1.00e-03 | train_loss 0.0124 | val_loss 0.0866 | val_macro_f1 0.3472
Epoch 005 | lr

# BiLSTM1 CrossEntropyLoss

In [ ]:
import numpy as np
from pathlib import Path
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

RESULT_ROOT = Path(r"C:\SeniorProject\Good_Data\NPZ_SPLITS_FILTERED_CAP1H_IMPD\BILSTM_RESULTS_STABLE")

pred_file = RESULT_ROOT / "test_predictions_balancedseq_thr.npz"

data = np.load(pred_file, allow_pickle=True)

y_true = data["y_true"]
y_pred = data["y_pred"]
prob_preictal = data["prob_preictal"]
bases = data["bases"]
end_wis = data["end_wis"]
used_threshold = float(data["used_threshold"][0])

# =========================================
# 1) Basic metrics
# =========================================
acc = accuracy_score(y_true, y_pred)
bal_acc = balanced_accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")
weighted_f1 = f1_score(y_true, y_pred, average="weighted")

# preictal-specific
prec_pre = precision_score(y_true == 1, y_pred == 1, zero_division=0)
rec_pre  = recall_score(y_true == 1, y_pred == 1, zero_division=0)
f1_pre   = f1_score(y_true == 1, y_pred == 1, zero_division=0)

print("\n===== OVERALL METRICS =====")
print("Accuracy           :", acc)
print("Balanced Accuracy  :", bal_acc)
print("Macro F1           :", macro_f1)
print("Weighted F1        :", weighted_f1)

print("\n===== PREICTAL-SPECIFIC METRICS =====")
print("Preictal Precision :", prec_pre)
print("Preictal Recall    :", rec_pre)
print("Preictal F1        :", f1_pre)

print("\n===== THRESHOLD USED =====")
print("Used threshold     :", used_threshold)

# =========================================
# 2) Per-class metrics
# =========================================
print("\n===== PER-CLASS REPORT =====")
print(classification_report(y_true, y_pred, digits=4))

# =========================================
# 3) Confusion Matrix
# =========================================
cm = confusion_matrix(y_true, y_pred)

print("\n===== CONFUSION MATRIX =====")
print(cm)

# =========================================
# 4) Recall لكل كلاس
# =========================================
recalls = recall_score(y_true, y_pred, average=None, zero_division=0)

for i, r in enumerate(recalls):
    print(f"Recall (Class {i}): {r:.4f}")

# =========================================
# 5) Precision لكل كلاس
# =========================================
precisions = precision_score(y_true, y_pred, average=None, zero_division=0)

for i, p in enumerate(precisions):
    print(f"Precision (Class {i}): {p:.4f}")

# =========================================
# 6) Preictal probability stats
# =========================================
print("\n===== PREICTAL PROBABILITY STATS =====")
print("Min prob :", float(prob_preictal.min()))
print("Max prob :", float(prob_preictal.max()))
print("Mean prob:", float(prob_preictal.mean()))

print("Preictal Recall   :", rec_pre)
print("Preictal Precision:", prec_pre)
print("Preictal F1       :", f1_pre)
print("Balanced Accuracy :", bal_acc)
print("Confusion Matrix:\n", cm)

# =========================================
# 7) Optional sample rows
# =========================================
print("\n===== SAMPLE PREDICTIONS (first 10) =====")
for i in range(min(10, len(y_true))):
    print(
        f"idx={i:04d} | base={bases[i]} | end_wi={end_wis[i]} | "
        f"true={y_true[i]} | pred={y_pred[i]} | prob_preictal={prob_preictal[i]:.6f}"
    )


===== OVERALL METRICS =====
Accuracy           : 0.8042399472755611
Balanced Accuracy  : 0.405625363609299
Macro F1           : 0.3125612819879296
Weighted F1        : 0.8782336231602186

===== PREICTAL-SPECIFIC METRICS =====
Preictal Precision : 0.017301890563915193
Preictal Recall    : 0.179131773991722
Preictal F1        : 0.03155587772733246

===== THRESHOLD USED =====
Used threshold     : 0.75

===== PER-CLASS REPORT =====
              precision    recall  f1-score   support

           0     0.9859    0.8143    0.8919   1387213
           1     0.0173    0.1791    0.0316     20778
           2     0.0073    0.2235    0.0142      1602

    accuracy                         0.8042   1409593
   macro avg     0.3369    0.4056    0.3126   1409593
weighted avg     0.9705    0.8042    0.8782   1409593


===== CONFUSION MATRIX =====
[[1129571  211252   46390]
 [  15036    3722    2020]
 [   1097     147     358]]
Recall (Class 0): 0.8143
Recall (Class 1): 0.1791
Recall (Class 2): 0.2235

# BiLSTM2 Focalloss

In [ ]:
from pathlib import Path
import numpy as np
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_recall_fscore_support,
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

# =========================================================
# PATH
# =========================================================
OUT_ROOT = Path(r"C:\SeniorProject\Good_Data\NPZ_SPLITS_FILTERED_CAP1H_IMPD\BILSTM_RESULTS_STABLE_FOCAL")

PRED_NPZ = OUT_ROOT / "test_predictions.npz"

CLASS_NAMES = ["Normal", "Preictal", "Ictal"]

# =========================================================
# LOAD
# =========================================================
data = np.load(PRED_NPZ, allow_pickle=True)

y_true = data["y_true"]
y_pred = data["y_pred"]
probs  = data["probs"]

print("Total samples:", len(y_true))

# =========================================================
# BASIC METRICS
# =========================================================
acc = accuracy_score(y_true, y_pred)
bal_acc = balanced_accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
weighted_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print("\n==============================")
print("OVERALL METRICS")
print("==============================")
print("Accuracy          :", acc)
print("Balanced Accuracy :", bal_acc)
print("Macro F1          :", macro_f1)
print("Weighted F1       :", weighted_f1)

# =========================================================
# CONFUSION MATRIX
# =========================================================
cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])

print("\n==============================")
print("CONFUSION MATRIX")
print("Rows=True, Cols=Pred")
print("==============================")
print(cm)

# =========================================================
# CLASS REPORT
# =========================================================
print("\n==============================")
print("CLASSIFICATION REPORT")
print("==============================")
print(classification_report(
    y_true,
    y_pred,
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
))

# =========================================================
# TP / FP / FN / TN
# =========================================================
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=[0, 1, 2], zero_division=0
)

print("\n==============================")
print("TP / FP / FN / TN PER CLASS")
print("==============================")

for i, name in enumerate(CLASS_NAMES):
    TP = cm[i, i]
    FP = cm[:, i].sum() - TP
    FN = cm[i, :].sum() - TP
    TN = cm.sum() - (TP + FP + FN)

    print(f"\n--- {name} ---")
    print("Support   :", support[i])
    print("TP        :", int(TP))
    print("FP        :", int(FP))
    print("FN        :", int(FN))
    print("TN        :", int(TN))
    print("Precision :", precision[i])
    print("Recall    :", recall[i])
    print("F1        :", f1[i])

# =========================================================
# PREICTAL ANALYSIS
# =========================================================
print("\n==============================")
print("PREICTAL ANALYSIS")
print("==============================")

idx = 1

TP = cm[idx, idx]
FP = cm[:, idx].sum() - TP
FN = cm[idx, :].sum() - TP
TN = cm.sum() - (TP + FP + FN)

print("TP        :", int(TP))
print("FP        :", int(FP))
print("FN        :", int(FN))
print("TN        :", int(TN))
print("Precision :", precision[idx])
print("Recall    :", recall[idx])
print("F1        :", f1[idx])

# مصدر الفولز بوزتف
print("\nFalse Positives breakdown:")
print("Normal -> Preictal :", int(cm[0, 1]))
print("Ictal  -> Preictal :", int(cm[2, 1]))

# =========================================================
# PREDICTION DISTRIBUTION
# =========================================================
print("\n==============================")
print("PREDICTED COUNTS")
print("==============================")
for i, name in enumerate(CLASS_NAMES):
    print(f"{name}: {(y_pred == i).sum()}")

print("\n==============================")
print("TRUE COUNTS")
print("==============================")
for i, name in enumerate(CLASS_NAMES):
    print(f"{name}: {(y_true == i).sum()}")

# =========================================================
# FALSE POSITIVE RATE (Preictal)
# =========================================================
print("\n==============================")
print("FALSE POSITIVE RATE (Preictal)")
print("==============================")

FP_pre = cm[:, 1].sum() - cm[1, 1]
FN_pre = cm[1, :].sum() - cm[1, 1]
TN_pre = cm.sum() - (cm[1, 1] + FP_pre + FN_pre)

FPR = FP_pre / (FP_pre + TN_pre + 1e-8)

print("FPR Preictal:", FPR)

Total samples: 5631850

OVERALL METRICS
Accuracy          : 0.7456002912009375
Balanced Accuracy : 0.6619237360594133
Macro F1          : 0.5392186348817797
Weighted F1       : 0.641630146007884

CONFUSION MATRIX
Rows=True, Cols=Pred
[[     80 1220144  167565]
 [      1 3865776   44019]
 [      0    1012  333253]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

      Normal     0.9877    0.0001    0.0001   1387789
    Preictal     0.7599    0.9887    0.8594   3909796
       Ictal     0.6117    0.9970    0.7582    334265

    accuracy                         0.7456   5631850
   macro avg     0.7864    0.6619    0.5392   5631850
weighted avg     0.8073    0.7456    0.6416   5631850


TP / FP / FN / TN PER CLASS

--- Normal ---
Support   : 1387789
TP        : 80
FP        : 1
FN        : 1387709
TN        : 4244060
Precision : 0.9876543209876543
Recall    : 5.764565074373698e-05
F1        : 0.00011528457276257863

--- Preictal ---
Support   : 3909796
TP      